# LNP-MFGO Phase 2B-R v3: GNN — A100 80GB Production (Merged)

## Merged from: A100 notebook + AllFixes notebook

### What this notebook takes from each:

| Feature | Source | Why |
|---|---|---|
| GPU-tier auto-detect (A100/L40/RTX/CPU) | A100 | Auto-configures batch/workers/compile/AMP per GPU |
| Batch=256, Workers=8, Seeds=5, Epochs=150 | A100 | Maximises A100 throughput |
| `torch.compile(mode='max-autotune')` | A100 | A100 kernel benchmarking |
| Hidden search [128,192,256], Layers [3,5] | A100 | 80GB VRAM headroom |
| 30 Optuna trials | A100 | Wider search space |
| Raw Z-norm target (dual target) | A100 | Ablation-ready |
| Pass/Fail benchmark checks | A100 | Manuscript-ready |
| JSON save of best params | A100 | Reproducibility |
| COLORS dict + styled plots | A100 | Publication figures |
| Transductive study-embedding fine-tune | AllFixes | Cold-start robustness (optional) |
| Dynamic TAB_COLS builder | AllFixes | Handles missing columns gracefully |
| Explicit FP16/BF16 branch in train loop | AllFixes | Correct scaler handling per dtype |
| `CrossComponentAttentionV3` naming | AllFixes | Cleaner class naming |

### Bug fixes applied (not in either notebook):
1. `scaler.unscale_()` before `clip_grad_norm_` in A100 loop — **already correct in A100**, added to AllFixes path
2. `RobustScaler` imported but never used in AllFixes — removed
3. `train_gnn` vs `train_gnn_v3` naming inconsistency — unified to `train_gnn`
4. `enum_cache` None-graph fallback — AllFixes uses `or smiles_to_graph(smi)` which can still be None; A100 uses explicit check — kept A100 pattern
5. Added `!pip install` cell as Cell 1 (run once)

### Files needed for A100 training:
1. **This notebook** (`.ipynb`)
2. **`LNP_Atlas_Bioactivity_Reextracted.csv`** — your main dataset

That's it. No other files needed.


## Cell 0 — Install Dependencies (run once per new env)

In [ ]:
# =============================================================
# RUN ONCE: Install all dependencies for A100
# =============================================================
# Uncomment and run this cell in a fresh environment.
# After first run, you can comment it out or skip it.

# ── Core ML stack (CUDA 12.1 — matches RunPod A100 images) ──
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# ── If your RunPod image already has PyTorch+CUDA, skip above and just do: ──
!pip install -q rdkit-pypi optuna lightgbm scikit-learn scipy pandas numpy matplotlib seaborn tqdm

# ── Verify GPU ──
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## 0 — Imports & A100 Configuration

In [ ]:
# ============================================================
# 0.1 — Imports & Configuration (A100-80GB Optimized)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, json, copy
from pathlib import Path
from collections import defaultdict

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from rdkit import Chem
from rdkit.Chem import AllChem

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print('⚠ pip install optuna')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('⚠ pip install lightgbm (optional, for ensemble comparison)')

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── GPU auto-detection (A100 / L40 / RTX / CPU) ──
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {GPU_NAME} ({GPU_MEM:.1f} GB)')
    USE_AMP = True

    if GPU_MEM >= 70:                          # A100 80GB
        BATCH_SIZE = 256
        AMP_DTYPE = torch.float16              # A100 FP16 tensor cores
        NUM_WORKERS = 8
        COMPILE_MODE = 'max-autotune'
        print(f'  → A100 profile: batch=256, FP16+GradScaler, workers=8, compile=max-autotune')
    elif GPU_MEM >= 40:                        # L40 48GB / A100 40GB
        BATCH_SIZE = 128
        AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        NUM_WORKERS = 4
        COMPILE_MODE = 'reduce-overhead'
        print(f'  → L40-class profile: batch=128, {AMP_DTYPE}, workers=4')
    elif GPU_MEM >= 20:                        # RTX 3090/4090
        BATCH_SIZE = 64
        AMP_DTYPE = torch.float16
        NUM_WORKERS = 4
        COMPILE_MODE = 'reduce-overhead'
        print(f'  → RTX profile: batch=64, FP16, workers=4')
    else:
        BATCH_SIZE = 32
        AMP_DTYPE = torch.float16
        NUM_WORKERS = 2
        COMPILE_MODE = 'default'
        print(f'  → Small GPU profile: batch=32')
else:
    DEVICE = torch.device('cpu')
    USE_AMP = False; AMP_DTYPE = torch.float32; BATCH_SIZE = 32
    NUM_WORKERS = 0; COMPILE_MODE = None
    print('⚠ CPU mode — will be very slow')

PERSISTENT_WORKERS = NUM_WORKERS > 0
OPTUNA_WORKERS = 0
OPTUNA_PERSISTENT = False

PROJECT_DIR = Path('.')
OUTPUT_DIR = PROJECT_DIR / 'outputs'; OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / 'figures'; FIGURE_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 11,
                      'font.family': 'sans-serif', 'axes.grid': True, 'grid.alpha': 0.3})
COLORS = {'primary':'#2563EB','secondary':'#DC2626','tertiary':'#059669',
          'quaternary':'#D97706','purple':'#7C3AED'}

print(f'\nSetup complete — device={DEVICE}, batch={BATCH_SIZE}, AMP={AMP_DTYPE}')


## 1 — Data Loading & Preparation

In [ ]:
# ============================================================
# 1.1 — Load Data, Log-Transform, Z-Normalise
# ============================================================

df_raw = pd.read_csv(PROJECT_DIR / 'LNP_Atlas_Bioactivity_Reextracted.csv',
                      encoding='latin-1', low_memory=False)
print(f'Loaded: {df_raw.shape}')

PS_COL = 'particle_size_nm_std_num'
df_raw['ps_log'] = np.log1p(df_raw[PS_COL])

mask_ps = df_raw['ps_log'].notna()
study_stats = df_raw[mask_ps].groupby('paper_doi')['ps_log'].agg(['mean','std'])
g_mean = df_raw[mask_ps]['ps_log'].mean()
g_std = df_raw[mask_ps]['ps_log'].std()
study_stats['std'] = study_stats['std'].fillna(g_std)
study_stats.loc[study_stats['std'] < 1e-6, 'std'] = g_std

df_raw['ps_smean'] = df_raw['paper_doi'].map(study_stats['mean']).fillna(g_mean)
df_raw['ps_sstd'] = df_raw['paper_doi'].map(study_stats['std']).fillna(g_std)
df_raw['target_znorm'] = (df_raw['ps_log'] - df_raw['ps_smean']) / df_raw['ps_sstd']

# Mean-centered target (less aggressive normalisation — ablation-ready)
df_raw['target_meancenter'] = df_raw['ps_log'] - df_raw['ps_smean']

# Raw-scale Z-norm (from A100 notebook — useful for ablation)
study_stats_raw = df_raw[df_raw[PS_COL].notna()].groupby('paper_doi')[PS_COL].agg(['mean','std'])
gr_mean = df_raw[df_raw[PS_COL].notna()][PS_COL].mean()
gr_std = df_raw[df_raw[PS_COL].notna()][PS_COL].std()
study_stats_raw['std'] = study_stats_raw['std'].fillna(gr_std)
study_stats_raw.loc[study_stats_raw['std'] < 1e-6, 'std'] = gr_std
df_raw['raw_smean'] = df_raw['paper_doi'].map(study_stats_raw['mean']).fillna(gr_mean)
df_raw['raw_sstd'] = df_raw['paper_doi'].map(study_stats_raw['std']).fillna(gr_std)
df_raw['target_raw_znorm'] = (df_raw[PS_COL] - df_raw['raw_smean']) / df_raw['raw_sstd']

print(f'Log Z-norm range: [{df_raw["target_znorm"].min():.2f}, {df_raw["target_znorm"].max():.2f}]')
print(f'Raw Z-norm range: [{df_raw["target_raw_znorm"].min():.2f}, {df_raw["target_raw_znorm"].max():.2f}]')


In [ ]:
# ============================================================
# 1.2 — Molecular Graph Construction + SMILES Enumeration
#        FIX #1: 7-dim BOND features
# ============================================================

ATOM_LIST = [6,7,8,9,15,16,17,35,53,0]
HYB_LIST = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2]
ATOM_DIM = 33
BOND_DIM = 7

def one_hot(val, allowed):
    enc = [0]*(len(allowed)+1)
    if val in allowed: enc[allowed.index(val)] = 1
    else: enc[-1] = 1
    return enc

def atom_feats(a):
    return (one_hot(a.GetAtomicNum(), ATOM_LIST) + one_hot(a.GetDegree(), [0,1,2,3,4,5])
            + one_hot(a.GetFormalCharge(), [-1,0,1,2]) + one_hot(a.GetNumRadicalElectrons(), [0,1])
            + one_hot(a.GetHybridization(), HYB_LIST) + [a.GetIsAromatic()])

BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

def bond_feats(b):
    bt = [0]*4
    if b.GetBondType() in BOND_TYPES:
        bt[BOND_TYPES.index(b.GetBondType())] = 1
    return bt + [int(b.GetIsConjugated()), int(b.IsInRing()),
                 int(b.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)]

def smiles_to_graph(smi):
    if pd.isna(smi) or not isinstance(smi, str) or len(smi) < 3: return None
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.FloatTensor([atom_feats(a) for a in mol.GetAtoms()])
    ei, ef = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        ei.extend([[i,j],[j,i]])
        bf = bond_feats(b)
        ef.extend([bf, bf])
    if not ei:
        return {'x': x, 'edge_index': torch.LongTensor([[0],[0]]),
                'edge_attr': torch.zeros(1, BOND_DIM), 'num_atoms': x.shape[0]}
    return {'x': x, 'edge_index': torch.LongTensor(ei).t().contiguous(),
            'edge_attr': torch.FloatTensor(ef), 'num_atoms': x.shape[0]}

def enumerate_smiles(smi, n_variants=5):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return [smi]
    variants = set()
    variants.add(Chem.MolToSmiles(mol))
    for _ in range(n_variants * 3):
        try:
            variants.add(Chem.MolToSmiles(mol, doRandom=True))
            if len(variants) >= n_variants: break
        except: pass
    return list(variants)[:n_variants]

LIPID_SMILES = ['ionizable_lipid_smiles','helper_lipid_smiles',
                'sterol_lipid_smiles','peg_lipid_smiles']
LIPID_NAMES = ['ionizable','helper','sterol','peg']

cache = {}
def get_g(s):
    if s not in cache: cache[s] = smiles_to_graph(s)
    return cache[s]

print('Building molecular graphs with bond features...')
all_graphs = {n: [] for n in LIPID_NAMES}
all_smiles = {n: [] for n in LIPID_NAMES}
valid_idx = []

for idx, row in df_raw.iterrows():
    ok = True; rg = {}; rs = {}
    for ln, sc in zip(LIPID_NAMES, LIPID_SMILES):
        smi = row[sc]
        g = get_g(smi) if pd.notna(smi) else None
        if g is None: ok = False; break
        rg[ln] = g; rs[ln] = str(smi)
    if ok:
        for n in LIPID_NAMES:
            all_graphs[n].append(rg[n])
            all_smiles[n].append(rs[n])
        valid_idx.append(idx)

df = df_raw.loc[valid_idx].reset_index(drop=True)

print('Pre-computing SMILES enumerations...')
N_AUGMENT = 5
enum_cache = {}
unique_smiles = set()
for n in LIPID_NAMES:
    unique_smiles.update(all_smiles[n])
print(f'  Unique SMILES: {len(unique_smiles)}')

for smi in unique_smiles:
    variants = enumerate_smiles(smi, N_AUGMENT)
    enum_cache[smi] = []
    for v in variants:
        g = smiles_to_graph(v)
        if g is not None: enum_cache[smi].append(g)
    if not enum_cache[smi]: enum_cache[smi] = [get_g(smi)]

print(f'Valid formulations: {len(df)}')

# ── Dynamic TAB_COLS (from AllFixes — handles missing columns) ──
TAB_COLS = []
for c in ['Ratio_Ionizable','Ratio_Helper','Ratio_Sterol','Ratio_PEG']:
    if c in df.columns: TAB_COLS.append(c)
for c in ['Process_FlowRate','Process_Ratio_AqOrg','Process_Is_Microfluidic',
          'Process_pH']:
    if c in df.columns and df[c].notna().mean() > 0.3:
        TAB_COLS.append(c)
print(f'Tabular features ({len(TAB_COLS)}): {TAB_COLS}')

tab_data = df[TAB_COLS].fillna(df[TAB_COLS].median()).values.astype(np.float32)

study_le = LabelEncoder()
df['study_id'] = study_le.fit_transform(df['paper_doi'].fillna('UNK'))
N_STUDIES = df['study_id'].nunique()
print(f'Studies: {N_STUDIES}, Bond dim: {BOND_DIM}')
print('Data ready')


## 2 — Model Architecture (v3: Bond Features + Residual GIN)

In [ ]:
# ============================================================
# 2.1 — GIN with Edge Features + Gated Residual Connections
# ============================================================

class GINLayerV3(nn.Module):
    def __init__(self, dim, bond_dim=BOND_DIM):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.BatchNorm1d(dim),
                                  nn.GELU(), nn.Linear(dim, dim), nn.GELU())
        self.edge_mlp = nn.Sequential(nn.Linear(bond_dim, dim), nn.GELU())
        self.gate = nn.Sequential(nn.Linear(dim * 2, 1), nn.Sigmoid())

    def forward(self, x, edge_index, edge_attr):
        row, col = edge_index
        edge_weight = self.edge_mlp(edge_attr)
        msg = x[row] * edge_weight
        agg = torch.zeros_like(x)
        agg.index_add_(0, col, msg)
        h_new = self.mlp((1 + self.eps) * x + agg)
        gate = self.gate(torch.cat([x, h_new], dim=-1))
        return gate * h_new + (1 - gate) * x


class SharedGINEncoderV3(nn.Module):
    def __init__(self, atom_dim=ATOM_DIM, bond_dim=BOND_DIM, type_dim=4,
                 hidden=128, out=128, n_layers=3, dropout=0.2):
        super().__init__()
        self.type_embedding = nn.Embedding(4, type_dim)
        self.atom_proj = nn.Linear(atom_dim + type_dim, hidden)
        self.layers = nn.ModuleList([GINLayerV3(hidden, bond_dim) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(hidden, out)

    def forward(self, x, edge_index, edge_attr, batch_vec, lipid_type_id):
        type_emb = self.type_embedding(
            torch.tensor(lipid_type_id, device=x.device))
        type_expanded = type_emb.unsqueeze(0).expand(x.size(0), -1)
        h = self.atom_proj(torch.cat([x, type_expanded], dim=-1))
        for layer in self.layers:
            h = self.dropout(layer(h, edge_index, edge_attr))
        n_graphs = batch_vec.max() + 1
        idx = batch_vec.unsqueeze(1).expand_as(h)
        pooled = torch.zeros(n_graphs, h.size(1), device=h.device, dtype=h.dtype)
        pooled.scatter_reduce_(0, idx, h, reduce='mean', include_self=False)
        return self.out_proj(pooled)


class CrossComponentAttentionV3(nn.Module):
    def __init__(self, dim=128, heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(dim*2, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        a, w = self.attn(x, x, x, need_weights=True, average_attn_weights=True)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x.mean(1), w


class LNPMFGO_v3(nn.Module):
    def __init__(self, atom_dim=ATOM_DIM, bond_dim=BOND_DIM, hidden=128,
                 n_layers=3, tab_dim=8, n_studies=60, study_emb=16, dropout=0.2):
        super().__init__()
        self.gin = SharedGINEncoderV3(atom_dim, bond_dim, 4, hidden, hidden, n_layers, dropout)
        self.cross_attn = CrossComponentAttentionV3(hidden, 4, dropout)
        self.tab_enc = nn.Sequential(nn.Linear(tab_dim, 64), nn.GELU(),
                                      nn.Dropout(dropout), nn.Linear(64, 32))
        self.study_emb = nn.Embedding(n_studies + 1, study_emb)
        fusion = hidden + 32 + study_emb
        self.head = nn.Sequential(
            nn.Linear(fusion, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1))

    def forward(self, graph_batch, tabular, study_ids):
        lipid_embs = []
        for i, n in enumerate(LIPID_NAMES):
            x, ei, ea, bv = graph_batch[n]
            emb = self.gin(x, ei, ea, bv, lipid_type_id=i)
            lipid_embs.append(emb)
        form_emb, attn_w = self.cross_attn(torch.stack(lipid_embs, dim=1))
        tab_emb = self.tab_enc(tabular)
        s_emb = self.study_emb(study_ids)
        fused = torch.cat([form_emb, tab_emb, s_emb], dim=-1)
        return self.head(fused).squeeze(-1), attn_w

_t = LNPMFGO_v3(n_studies=N_STUDIES if 'N_STUDIES' in dir() else 60)
print(f'LNP-MFGO v3: {sum(p.numel() for p in _t.parameters()):,} parameters')
del _t
print('Model v3 defined (bond features + residual gates)')


## 3 — Dataset & Training (A100 Optimized)

In [ ]:
# ============================================================
# 3.1 — Dataset with edge_attr in collate
# ============================================================

class LNPDatasetV3(Dataset):
    def __init__(self, indices, canonical_graphs, smiles_dict, tab, targets,
                 study_ids, enum_cache, augment=True):
        self.indices = indices
        self.can_graphs = canonical_graphs
        self.smiles = smiles_dict
        self.tab = tab
        self.targets = targets
        self.study_ids = study_ids
        self.enum_cache = enum_cache
        self.augment = augment

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        graphs = {}
        for n in LIPID_NAMES:
            if self.augment:
                smi = self.smiles[n][i]
                variants = self.enum_cache.get(smi, [self.can_graphs[n][i]])
                graphs[n] = variants[np.random.randint(len(variants))]
            else:
                graphs[n] = self.can_graphs[n][i]
        return graphs, self.tab[idx], self.targets[idx], self.study_ids[idx]


def collate_fn_v3(batch):
    gs_list, tab_list, tgt_list, sid_list = zip(*batch)
    gb = {}
    for n in LIPID_NAMES:
        xs, eis, eas, batch_ids = [], [], [], []
        off = 0
        for graph_idx, sg in enumerate(gs_list):
            g = sg[n]
            na = g['num_atoms']
            xs.append(g['x'])
            eis.append(g['edge_index'] + off)
            eas.append(g['edge_attr'])
            batch_ids.append(torch.full((na,), graph_idx, dtype=torch.long))
            off += na
        gb[n] = (torch.cat(xs), torch.cat(eis, 1),
                 torch.cat(eas, 0), torch.cat(batch_ids))
    return (gb,
            torch.FloatTensor(np.array(tab_list)),
            torch.FloatTensor(np.array(tgt_list)),
            torch.LongTensor(np.array(sid_list)))

print('Dataset v3 ready (edge_attr in collate)')


In [ ]:
# ============================================================
# 3.2 — Training Loop (A100: no grad accum, FP16 GradScaler)
#        Merged: correct scaler branch from AllFixes +
#                unscale_ before clip from A100
# ============================================================

def to_dev(gb, tab, tgt, sid):
    gd = {}
    for n in LIPID_NAMES:
        gd[n] = (gb[n][0].to(DEVICE, non_blocking=True),
                 gb[n][1].to(DEVICE, non_blocking=True),
                 gb[n][2].to(DEVICE, non_blocking=True),
                 gb[n][3].to(DEVICE, non_blocking=True))
    return (gd, tab.to(DEVICE, non_blocking=True),
            tgt.to(DEVICE, non_blocking=True),
            sid.to(DEVICE, non_blocking=True))


def train_gnn(model, train_ld, val_ld, epochs=150, lr=5e-4,
              patience=30, warmup_epochs=5):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    huber = nn.HuberLoss(delta=1.0)

    # GradScaler only needed for FP16 (not BF16)
    use_scaler = USE_AMP and AMP_DTYPE == torch.float16
    scaler = GradScaler(DEVICE.type, enabled=use_scaler)

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val = float('inf'); best_state = None; pat = 0

    for epoch in range(epochs):
        model.train()
        for gb, tab, tgt, sid in train_ld:
            optimizer.zero_grad(set_to_none=True)
            gd, t, y, s = to_dev(gb, tab, tgt, sid)
            with autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                pred, _ = model(gd, t, s)
                loss = huber(pred, y)

            if use_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        scheduler.step()

        model.eval()
        vl = 0; nv = 0
        with torch.no_grad():
            for gb, tab, tgt, sid in val_ld:
                gd, t, y, s = to_dev(gb, tab, tgt, sid)
                with autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    pred, _ = model(gd, t, s)
                    vl += huber(pred, y).item(); nv += 1
        vl /= max(nv, 1)

        if vl < best_val:
            best_val = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= patience: break

    model.load_state_dict(best_state)
    return model, epoch + 1


def finetune_study_embedding(model, test_loader, n_steps=10, lr=1e-3):
    """Transductive fine-tuning of study embedding only (from AllFixes).
    Freeze all params except study_emb, minimise prediction variance.
    Optional — uncomment in LOSO loop to enable."""
    base = model._orig_mod if hasattr(model, '_orig_mod') else model
    for p in base.parameters():
        p.requires_grad = False
    base.study_emb.weight.requires_grad = True
    opt = optim.Adam([base.study_emb.weight], lr=lr)
    for step in range(n_steps):
        for gb, tab, tgt, sid in test_loader:
            opt.zero_grad()
            gd, t, y, s = to_dev(gb, tab, tgt, sid)
            with autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                pred, _ = model(gd, t, s)
                loss = pred.var()
            loss.backward()
            opt.step()
    for p in base.parameters():
        p.requires_grad = True
    return model

print('Training loop ready (A100: FP16 GradScaler, no accum, transductive fine-tune available)')


## 4 — Optuna GNN Tuning (A100: wider search, more trials)

In [ ]:
# ============================================================
# 4.1 — Optuna Tuning (Fixed for Compilation Thrashing & CPU Bottleneck)
# ============================================================

mask_t = df['target_znorm'].notna()
df_model = df[mask_t].reset_index(drop=True)
valid_t_idx = df[mask_t].index.tolist()
graphs_f = {n: [all_graphs[n][i] for i in valid_t_idx] for n in LIPID_NAMES}
smiles_f = {n: [all_smiles[n][i] for i in valid_t_idx] for n in LIPID_NAMES}
tab_f = tab_data[valid_t_idx]
tgt_f = df_model['target_znorm'].values.astype(np.float32)
sid_f = df_model['study_id'].values
studies_f = df_model['paper_doi']

study_counts = studies_f.value_counts()
usable = study_counts[study_counts >= 5].index.tolist()
print(f'Samples: {len(df_model)}, Usable LOSO studies: {len(usable)}')

# --- THE FIXES ARE HERE ---
# Override the Cell 0 settings to ensure data loaders aren't bottlenecked
OPTUNA_WORKERS = min(4, NUM_WORKERS if 'NUM_WORKERS' in locals() else 4)
OPTUNA_PERSISTENT = OPTUNA_WORKERS > 0

# L40 / A100-appropriate settings
OPTUNA_FOLDS = 4
OPTUNA_EPOCHS = 60
OPTUNA_PATIENCE = 15
OPTUNA_TRIALS = 30

def quick_inner_cv(hidden, n_layers, lr, dropout, n_folds=OPTUNA_FOLDS):
    large = study_counts[study_counts >= 15].nlargest(2).index.tolist()
    med_pool = study_counts[(study_counts >= 8) & (study_counts < 15)]
    small = med_pool.sample(min(2, len(med_pool)), random_state=SEED).index.tolist()
    test_studies = (large + small)[:n_folds]

    r2s = []
    for test_study in test_studies:
        test_m = (studies_f.values == test_study)
        all_train_idx = np.where(~test_m)[0]
        te_idx = np.where(test_m)[0]
        if len(te_idx) < 3: continue

        train_studies = studies_f.iloc[all_train_idx].unique()
        np.random.seed(SEED)
        np.random.shuffle(train_studies)
        n_val_studies = max(2, len(train_studies) // 10)
        val_study_set = set(train_studies[:n_val_studies])
        val_mask = studies_f.iloc[all_train_idx].isin(val_study_set).values
        val_idx = all_train_idx[val_mask]
        train_idx = all_train_idx[~val_mask]

        sc = StandardScaler()
        t_tr = sc.fit_transform(tab_f[train_idx])
        t_va = sc.transform(tab_f[val_idx])
        t_te = sc.transform(tab_f[te_idx])

        ds_tr = LNPDatasetV3(train_idx, graphs_f, smiles_f, t_tr, tgt_f[train_idx],
                              sid_f[train_idx], enum_cache, augment=True)
        ds_va = LNPDatasetV3(val_idx, graphs_f, smiles_f, t_va, tgt_f[val_idx],
                              sid_f[val_idx], enum_cache, augment=False)
        
        # Use multiprocessing context if workers > 0 to prevent CUDA initialization errors
        _mp_ctx = 'spawn' if OPTUNA_WORKERS > 0 else None
        
        ld_tr = DataLoader(ds_tr, BATCH_SIZE, shuffle=True, collate_fn=collate_fn_v3,
                           pin_memory=True, num_workers=OPTUNA_WORKERS,
                           persistent_workers=OPTUNA_PERSISTENT, multiprocessing_context=_mp_ctx)
        ld_va = DataLoader(ds_va, BATCH_SIZE, shuffle=False, collate_fn=collate_fn_v3,
                           pin_memory=True, num_workers=OPTUNA_WORKERS,
                           persistent_workers=OPTUNA_PERSISTENT, multiprocessing_context=_mp_ctx)

        model = LNPMFGO_v3(hidden=hidden, n_layers=n_layers, tab_dim=len(TAB_COLS),
                           n_studies=N_STUDIES, study_emb=16, dropout=dropout).to(DEVICE)
        
        # --- THE COMPILE FIX ---
        try:
            # dynamic=True prevents PyTorch from recompiling on every batch size/shape change
            model = torch.compile(model, dynamic=True)
        except Exception: 
            pass # Failsafe: drop back to eager mode if compile fails

        model, _ = train_gnn(model, ld_tr, ld_va, epochs=OPTUNA_EPOCHS,
                              lr=lr, patience=OPTUNA_PATIENCE)

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for gb, tab, tgt, sid in DataLoader(
                LNPDatasetV3(te_idx, graphs_f, smiles_f, t_te, tgt_f[te_idx],
                             sid_f[te_idx], enum_cache, augment=False),
                BATCH_SIZE, shuffle=False, collate_fn=collate_fn_v3, pin_memory=True, 
                num_workers=OPTUNA_WORKERS, multiprocessing_context=_mp_ctx):
                gd, t, y, s = to_dev(gb, tab, tgt, sid)
                p, _ = model(gd, t, s)
                preds.append(p.float().cpu().numpy())
                trues.append(y.float().cpu().numpy())
        preds = np.concatenate(preds); trues = np.concatenate(trues)
        m = np.isfinite(trues) & np.isfinite(preds)
        if m.sum() >= 3: r2s.append(max(r2_score(trues[m], preds[m]), -1.0))
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return np.mean(r2s) if r2s else -1.0


if HAS_OPTUNA:
    def objective(trial):
        hidden = trial.suggest_categorical('hidden', [128, 192, 256])
        n_layers = trial.suggest_int('n_layers', 3, 5)
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        dropout = trial.suggest_float('dropout', 0.1, 0.4)
        return quick_inner_cv(hidden, n_layers, lr, dropout, n_folds=OPTUNA_FOLDS)

    print(f'Optuna: {OPTUNA_TRIALS} trials x {OPTUNA_FOLDS} folds x {OPTUNA_EPOCHS} ep (Workers: {OPTUNA_WORKERS})')
    study_opt = optuna.create_study(direction='maximize',
                                     sampler=optuna.samplers.TPESampler(seed=SEED))
    study_opt.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    best_gnn_params = study_opt.best_params
    print(f'Best: {best_gnn_params} (R2={study_opt.best_value:.4f})')
else:
    best_gnn_params = {'hidden': 192, 'n_layers': 4, 'lr': 5e-4, 'dropout': 0.2}
    print(f'Defaults: {best_gnn_params}')

with open(OUTPUT_DIR / 'best_gnn_params_v3.json', 'w') as f:
    json.dump(best_gnn_params, f, indent=2)
print('Tuning complete')

## 5 — LOSO Cross-Validation (A100: 5-seed, 150 epochs, all fixes)
- **FIX #4**: Study-aware validation
- **FIX #5**: Attention collected from ALL batches
- **FIX #6**: Prediction uncertainty
- **FIX #7**: Transductive study embedding (optional, commented out)

In [ ]:
# ============================================================
# 5.1 — LOSO-CV (A100: 5-Seed, 150 Epochs)
# ============================================================

N_SEEDS = 5
EPOCHS = 150
PATIENCE = 30

H = best_gnn_params['hidden']
NL = best_gnn_params['n_layers']
LR = best_gnn_params['lr']
DO = best_gnn_params['dropout']

print(f'Config: hidden={H}, layers={NL}, lr={LR:.4f}, dropout={DO:.2f}')
print(f'Seeds: {N_SEEDS}, Epochs: {EPOCHS}, Patience: {PATIENCE}, Batch: {BATCH_SIZE}')
print(f'Running {len(usable)} LOSO folds x {N_SEEDS} seeds...')

fold_results = []
all_attn = []
t_start = time.time()

for fold_i, test_study in enumerate(usable):
    test_m = (studies_f.values == test_study)
    all_nontest_idx = np.where(~test_m)[0]
    te_idx = np.where(test_m)[0]
    if len(te_idx) < 3 or len(all_nontest_idx) < 20: continue

    nontest_studies = studies_f.iloc[all_nontest_idx].unique()
    np.random.seed(SEED + fold_i)
    np.random.shuffle(nontest_studies)
    n_val_studies = max(2, len(nontest_studies) // 10)
    val_study_set = set(nontest_studies[:n_val_studies])
    val_mask = studies_f.iloc[all_nontest_idx].isin(val_study_set).values
    val_idx = all_nontest_idx[val_mask]
    train_idx = all_nontest_idx[~val_mask]

    sc = StandardScaler()
    t_tr = sc.fit_transform(tab_f[train_idx])
    t_va = sc.transform(tab_f[val_idx])
    t_te = sc.transform(tab_f[te_idx])

    test_sid = np.full(len(te_idx), N_STUDIES)

    ds_tr = LNPDatasetV3(train_idx, graphs_f, smiles_f, t_tr, tgt_f[train_idx],
                          sid_f[train_idx], enum_cache, augment=True)
    ds_va = LNPDatasetV3(val_idx, graphs_f, smiles_f, t_va, tgt_f[val_idx],
                          sid_f[val_idx], enum_cache, augment=False)
    ds_te = LNPDatasetV3(te_idx, graphs_f, smiles_f, t_te, tgt_f[te_idx],
                          test_sid, enum_cache, augment=False)

    _mp_ctx = 'spawn' if NUM_WORKERS > 0 else None
    ld_tr = DataLoader(ds_tr, BATCH_SIZE, shuffle=True, collate_fn=collate_fn_v3,
                       pin_memory=True, drop_last=False,
                       num_workers=NUM_WORKERS, persistent_workers=PERSISTENT_WORKERS,
                       multiprocessing_context=_mp_ctx)
    ld_va = DataLoader(ds_va, BATCH_SIZE, shuffle=False, collate_fn=collate_fn_v3,
                       pin_memory=True, num_workers=NUM_WORKERS,
                       persistent_workers=PERSISTENT_WORKERS,
                       multiprocessing_context=_mp_ctx)
    ld_te = DataLoader(ds_te, BATCH_SIZE, shuffle=False, collate_fn=collate_fn_v3,
                       pin_memory=True, num_workers=NUM_WORKERS,
                       persistent_workers=PERSISTENT_WORKERS,
                       multiprocessing_context=_mp_ctx)

    seed_preds = []
    seed_epochs = []
    fold_attns = []

    for seed_i in range(N_SEEDS):
        torch.manual_seed(SEED * 100 + fold_i * 10 + seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED * 100 + fold_i * 10 + seed_i)

        model = LNPMFGO_v3(hidden=H, n_layers=NL, tab_dim=len(TAB_COLS),
                           n_studies=N_STUDIES, study_emb=16, dropout=DO).to(DEVICE)

        with torch.no_grad():
            model.study_emb.weight[N_STUDIES] = model.study_emb.weight[:N_STUDIES].mean(0)

        try:
            model = torch.compile(model, mode=COMPILE_MODE, dynamic=True)
        except: pass

        model, n_ep = train_gnn(model, ld_tr, ld_va, EPOCHS, LR, PATIENCE)
        seed_epochs.append(n_ep)

        with torch.no_grad():
            unique_train = np.unique(sid_f[train_idx])
            base = model._orig_mod if hasattr(model, '_orig_mod') else model
            base.study_emb.weight[N_STUDIES] = base.study_emb.weight[unique_train].mean(0)

        # Optional: transductive fine-tuning (uncomment to enable)
        # model = finetune_study_embedding(model, ld_te, n_steps=10)

        model.eval()
        preds_s = []; trues_s = []; attns_s = []
        with torch.no_grad():
            for gb, tab, tgt, sid in ld_te:
                gd, t, y, s = to_dev(gb, tab, tgt, sid)
                with autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    p, aw = model(gd, t, s)
                preds_s.append(p.float().cpu().numpy())
                trues_s.append(y.float().cpu().numpy())
                attns_s.append(aw.float().cpu().numpy())

        seed_preds.append(np.concatenate(preds_s))
        trues_final = np.concatenate(trues_s)
        if attns_s:
            fold_attns.append(np.concatenate(attns_s, axis=0))
        del model

    ensemble_pred = np.mean(seed_preds, axis=0)
    ensemble_std = np.std(seed_preds, axis=0)

    if fold_attns:
        fold_attn_mean = np.concatenate(fold_attns, axis=0).mean(axis=0)
        all_attn.append(fold_attn_mean)

    yt, yp = trues_final, ensemble_pred
    m = np.isfinite(yt) & np.isfinite(yp)
    yt, yp, ystd = yt[m], yp[m], ensemble_std[m]

    if len(yt) >= 3:
        r2 = r2_score(yt, yp)
        rho = spearmanr(yt, yp)[0]
        rmse = np.sqrt(mean_squared_error(yt, yp))
        mae = mean_absolute_error(yt, yp)
    else:
        r2, rho, rmse, mae = np.nan, np.nan, np.nan, np.nan

    fold_results.append({
        'study': test_study, 'R2': r2, 'RMSE': rmse, 'MAE': mae,
        'Spearman_rho': rho, 'N': len(yt),
        'mean_epochs': np.mean(seed_epochs), 'train_size': len(train_idx),
        'pred_uncertainty_mean': ystd.mean(),
        'pred_uncertainty_max': ystd.max(),
    })

    elapsed = time.time() - t_start
    eta = elapsed / (fold_i+1) * (len(usable)-fold_i-1)
    r2_str = f'{r2:+.4f}' if np.isfinite(r2) else 'N/A'
    rho_str = f'{rho:+.3f}' if np.isfinite(rho) else 'N/A'
    print(f'  Fold {fold_i+1:2d}/{len(usable)}  R2={r2_str}  rho={rho_str}  '
          f'n={len(yt)}  unc={ystd.mean():.3f}  ({np.mean(seed_epochs):.0f}ep x{N_SEEDS}, ETA {eta/60:.0f}min)')

print(f'\nDone: {len(fold_results)} folds, {(time.time()-t_start)/60:.1f} min')


## 6 — Results & Benchmarks

In [ ]:
# ============================================================
# 6.1 — Summary & Benchmark
# ============================================================

rdf = pd.DataFrame(fold_results)
r2s = rdf['R2'].dropna()
rhos = rdf['Spearman_rho'].dropna()

print('GNN v3 (A100 | BOND FEATS | RESIDUAL | 5-SEED | STUDY-AWARE VAL)')
print('='*70)
print(f'  Mean R2:          {r2s.mean():+.4f} +/- {r2s.std():.4f}')
print(f'  Median R2:        {r2s.median():+.4f}')
print(f'  % Positive R2:    {100*(r2s>0).mean():.1f}%')
print(f'  Mean Spearman:    {rhos.mean():+.4f} +/- {rhos.std():.4f}')
print(f'  Best fold:        R2={r2s.max():+.4f}')
print(f'  Worst fold:       R2={r2s.min():+.4f}')
print(f'  Mean uncertainty: {rdf["pred_uncertainty_mean"].mean():.4f}')

weights = rdf['N'].values / rdf['N'].sum()
print(f'  Weighted R2:      {np.average(r2s, weights=weights[:len(r2s)]):+.4f}')
print(f'  Weighted Spearman:{np.average(rhos.values, weights=weights[:len(rhos)]):+.4f}')

print(f'\n{"="*70}')
print('BENCHMARK COMPARISON')
print(f'{"="*70}')
print(f'{"Approach":45s} {"R2":>8s} {"rho":>8s} {"% Pos":>6s}')
print(f'{"-"*70}')
print(f'  {">>> GNN v3 A100 (this notebook)":43s} {r2s.mean():+8.4f} {rhos.mean():+8.4f} {100*(r2s>0).mean():5.1f}%')
print(f'  {"Phase 2B-R GNN v2 (L40)":43s} {"~0.000":>8s} {"~0.15":>8s} {"~30":>5s}%')
print(f'  {"Phase 2A LightGBM (tuned)":43s} {0.018:+8.4f} {0.124:+8.4f} {38.5:5.1f}%')
print(f'  {"Phase 2A XGBoost (tuned)":43s} {0.015:+8.4f} {0.180:+8.4f} {46.2:5.1f}%')
print(f'  {"Phase 2A RF (tuned)":43s} {0.002:+8.4f} {0.178:+8.4f} {51.3:5.1f}%')

checks = [
    ('Mean R2 > +0.030', r2s.mean() > 0.030),
    ('Median R2 > 0.000', r2s.median() > 0.000),
    ('Mean Spearman > +0.200', rhos.mean() > 0.200),
    ('% Positive > 50%', (r2s > 0).mean() > 0.50),
    ('R2 std < 0.5', r2s.std() < 0.5),
]
print(f'\n{"="*70}')
for desc, passed in checks:
    print(f'  {"PASS" if passed else "FAIL"}  {desc}')
print(f'  Result: {sum(p for _,p in checks)}/{len(checks)} passed')

rdf.to_csv(OUTPUT_DIR / 'phase2b_v3_a100_fold_results.csv', index=False)
print(f'Saved: phase2b_v3_a100_fold_results.csv')


In [ ]:
# ============================================================
# 6.2 — Figures: Per-Fold R2 + Corrected Attention Heatmap
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sorted_r2 = rdf.sort_values('R2', ascending=False)
colors_b = [COLORS['tertiary'] if r > 0 else COLORS['secondary'] for r in sorted_r2['R2']]
ax.barh(range(len(sorted_r2)), sorted_r2['R2'].values, color=colors_b, alpha=0.8)
ax.set_yticks(range(len(sorted_r2)))
ax.set_yticklabels([str(s).split('/')[-1][:18] for s in sorted_r2['study']], fontsize=7)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Test R2')
ax.set_title(f'GNN v3 A100 LOSO-CV ({N_SEEDS}-seed, Huber, Bond Feats)\n'
             f'Mean R2={r2s.mean():+.4f}, {100*(r2s>0).mean():.0f}% positive', fontweight='bold')
ax.invert_yaxis()

ax = axes[1]
if all_attn:
    mean_attn = np.stack(all_attn).mean(0)
    sns.heatmap(mean_attn, xticklabels=['Ionizable','Helper','Sterol','PEG'],
                yticklabels=['Ionizable','Helper','Sterol','PEG'],
                annot=True, fmt='.3f', cmap='Blues', ax=ax, vmin=0)
    ax.set_title('Cross-Component Attention v3\n(all test samples)', fontweight='bold')

    col_sums = mean_attn.sum(axis=0)
    print('\nAttention received (column sums):')
    for name, cs in zip(['Ionizable','Helper','Sterol','PEG'], col_sums):
        print(f'  {name}: {cs:.3f}')

    names = ['Ionizable','Helper','Sterol','PEG']
    off_diag = []
    for i in range(4):
        for j in range(4):
            if i != j: off_diag.append((names[i], names[j], mean_attn[i,j]))
    off_diag.sort(key=lambda x: -x[2])
    print('Top off-diagonal:')
    for src, tgt, w in off_diag[:3]:
        print(f'  {src} -> {tgt}: {w:.3f}')
    np.save(OUTPUT_DIR / 'attention_weights_v3_a100.npy', mean_attn)
else:
    ax.text(0.5, 0.5, 'No attention data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'fig_gnn_v3_a100_results.png', bbox_inches='tight')
plt.show()
print('Saved figures')


In [ ]:
# ============================================================
# 6.3 — LightGBM Ensemble (FIX #8: proper per-fold scaling)
# ============================================================

if HAS_LGB:
    mordred_pruned_local = []
    mc = [c for c in df.columns if c.startswith('mordred_')]
    if mc:
        mdf_l = df[mc]
        zv = mdf_l.columns[mdf_l.var() == 0].tolist()
        mdf_l = mdf_l.drop(columns=zv)
        nzv_l = [c for c in mdf_l.columns if mdf_l[c].value_counts(normalize=True).iloc[0] > 0.95]
        mdf_l = mdf_l.drop(columns=nzv_l)
        mordred_pruned_local = mdf_l.columns.tolist()

    rdkit_c = ['MW','LogP','TPSA','HBD','HBA','RotBonds','HeavyAtoms','Rings',
               'AromaticRings','FractionCSP3','FormalCharge','NumAmines','NumAmides']

    for c in ['helper_lipid','sterol_lipid','peg_lipid']:
        if f'{c}_enc' not in df.columns:
            df[f'{c}_enc'] = LabelEncoder().fit_transform(df[c].fillna('UNK').astype(str))

    lip_e = ['helper_lipid_enc','sterol_lipid_enc','peg_lipid_enc']
    avail = [c for c in rdkit_c + mordred_pruned_local + TAB_COLS + lip_e if c in df_model.columns]
    desc_all = df_model[avail].fillna(df_model[avail].median()).values.astype(np.float32)

    lgb_params = {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.012,
                  'subsample': 0.6, 'colsample_bytree': 0.36, 'reg_alpha': 0.03,
                  'reg_lambda': 6.6, 'min_child_samples': 14, 'num_leaves': 22}

    print('Running LightGBM LOSO with proper scaling...')
    ensemble_results = []

    for fold_i, test_study in enumerate(usable):
        test_m = (studies_f.values == test_study)
        tr_idx = np.where(~test_m)[0]; te_idx = np.where(test_m)[0]
        if len(te_idx) < 3: continue

        sc_lgb = StandardScaler()
        X_tr = sc_lgb.fit_transform(desc_all[tr_idx])
        X_te = sc_lgb.transform(desc_all[te_idx])
        y_tr = tgt_f[tr_idx]; y_te = tgt_f[te_idx]

        lgb_model = lgb.LGBMRegressor(**lgb_params, random_state=SEED, verbose=-1)
        lgb_model.fit(X_tr, y_tr)
        lgb_pred = lgb_model.predict(X_te)

        m = np.isfinite(y_te) & np.isfinite(lgb_pred)
        lgb_r2 = r2_score(y_te[m], lgb_pred[m]) if m.sum() >= 3 else np.nan
        lgb_rho = spearmanr(y_te[m], lgb_pred[m])[0] if m.sum() >= 3 else np.nan

        gnn_fold = rdf[rdf['study'] == test_study]
        gnn_r2 = gnn_fold['R2'].values[0] if len(gnn_fold) > 0 else np.nan
        gnn_rho = gnn_fold['Spearman_rho'].values[0] if len(gnn_fold) > 0 else np.nan

        ensemble_results.append({
            'study': test_study, 'LGB_R2': lgb_r2, 'LGB_rho': lgb_rho,
            'GNN_R2': gnn_r2, 'GNN_rho': gnn_rho, 'N': int(m.sum()),
        })

    edf = pd.DataFrame(ensemble_results)
    print(f'\nLGB (scaled): Mean R2={edf["LGB_R2"].dropna().mean():+.4f}, '
          f'Mean rho={edf["LGB_rho"].dropna().mean():+.4f}')
    print(f'GNN v3:       Mean R2={edf["GNN_R2"].dropna().mean():+.4f}')

    valid_both = edf.dropna(subset=['LGB_R2','GNN_R2'])
    gnn_wins = (valid_both['GNN_R2'] > valid_both['LGB_R2']).sum()
    lgb_wins = (valid_both['LGB_R2'] > valid_both['GNN_R2']).sum()
    print(f'Head-to-head: GNN wins {gnn_wins}/{len(valid_both)}, LGB wins {lgb_wins}/{len(valid_both)}')
    edf.to_csv(OUTPUT_DIR / 'phase2b_v3_a100_ensemble.csv', index=False)
    print('Saved ensemble comparison')
else:
    print('LightGBM not installed')
